# 📚 Chapter 3. Welcome to LangChain (Ollama) 총정리

이 노트북은 LangChain의 가장 뼈대가 되는 **"프롬프트 제작 ➡️ AI 연산 ➡️ 결과물 출력 ➡️ 파이프라인 연결"**이라는 핵심 작업 흐름을 배우는 기초 단계입니다.

### 🎯 핵심 요약 (실무 개발 시 언제 꺼내봐야 할까?)

#### 1. 메시지 객체로 대화하기 (Predict Messages)
단순 텍스트 대신 대화의 역할을 **명확한 객체(Object)**로 분리하여 챗봇 모델(LLM)에게 전달하는 랭체인의 기초 작동법입니다.
* **`SystemMessage`**: AI의 성격, 규칙, 역할을 사전에 강력하게 부여하는 뒷배경 설정 공간.
* **`HumanMessage`**: 사용자가 실제로 텍스트 창에 입력하는 진짜 질문 공간.
* 이 객체들을 결합해서 `invoke`로 보내면, 누가 어떤 목적으로 말했는지 AI가 완벽하게 상황을 인지하고 대답해 줍니다.

#### 2. 프롬프트 템플릿 (Prompt Templates)
매번 비슷한 질문을 작성할 필요 없이, 빈칸(변수)만 쏙쏙 뚫어 놓은 **템플릿(빵틀)**을 만드는 유지보수 기술입니다.
* **`ChatPromptTemplate`**: 최신 채팅형 AI 모델들을 위해, `("system", "넌 수학자야")`, `("human", "{question}을 풀어줘")`처럼 구조화시켜 한 번에 묶어서 보내는 **필수 표준 템플릿 양식**입니다.

#### 3. ⭐️ 랭체인의 백미: 출력 파서와 파이프라인 (OutputParser & LCEL)
* **`StrOutputParser`**: AI가 반환한 크고 무거운 `AIMessage` 상자(각종 요금 영수증, 메타데이터 포함) 속에서, 우리가 화면에 띄울 **알맹이 내용물(문자열 텍스트)**만을 쏙 빼내서 깨끗하게 넘겨주는 필수 필터망입니다.
* **`LCEL (LangChain Expression Language)`**: 템플릿(`prompt`), AI 두뇌(`chat`), 그리고 필터망(`parser`)을 수학 기호인 **수직선(`|` 파이프)** 하나만 사용하여 길다란 벨트(조립 공정 라인)로 결합시키는 혁신적인 문법입니다. (`chain = prompt | chat | StrOutputParser()`)

#### 4. 더 깊은 체인 연계 공정 (Chaining Chains)
질문 하나로 도저히 끝나지 않는 복잡한 2단계 업무가 있을 때, **A 공정(Chain1)을 돌린 결과값을 무심하게 바로 B 공정(Chain2)의 재료(입력 파라미터)로 집어넣는 체인 콤보 기능**입니다. (`{"A의_결과물": chain1} | chain2`) 랭체인(LangChain)이라는 이름의 진정한 의미가 드러나는 핵심 설계법입니다! 

In [ ]:
# 3.0 LLMs and Chat Models
from langchain_ollama import OllamaLLM, ChatOllama

llm = OllamaLLM(model="llama3.3")
chat = ChatOllama(model="llama3.3")

a = llm.invoke("How many planets are there?")
b = chat.invoke("How many planets are there?")

print(f"LLM 결과: {a}")
print(f"Chat 결과: {b.content}") # Chat 모델은 객체를 반환하므로 .content가 필요합니다

LLM 결과: There are 8 planets in our solar system. 

1. Mercury
2. Mars
3. Venus
4. Earth
5. Neptune
6. Uranus
7. Saturn 
8. Jupiter
Chat 결과: There are 8 planets in our solar system. 

1. Mercury
2. Mars
3. Venus
4. Earth
5. Neptune
6. Uranus
7. Saturn
8. Jupiter


In [4]:
# 3.1 Predict Messages
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

chat = ChatOllama(model="llama3.3")

messages = [
    SystemMessage(content="You are a geography expert. And you only reply in Italian."),
    AIMessage(content="Ciao, mi chiamo Paolo!"),
    HumanMessage(content="What is the distance between Mexico and Thailand. Also, what is your name?")
]

chat.invoke(messages).content


"Il mio nome è Marco Rossi! La distanza tra il Messico e la Tailandia è di circa 17.300 chilometri. Il Messico si trova nell'emisfero occidentale, mentre la Tailandia si trova nell'emisfero orientale, quindi ci sono molti chilometri di oceano tra i due paesi. Per essere più precisi, la distanza è misurata in linea d'aria e può variare a seconda delle rotte aeree o marittime utilizzate. Spero che questo ti sia stato utile!"

In [7]:
# 3.2 Prompt Templates
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

chat = ChatOllama(
  model="llama3.3",
  temperature=0.1
)

template = PromptTemplate.from_template("What is the distance between {country_a} and {country_b}")
prompt = template.format(country_a="Mexico", country_b="Thailand")

chat.invoke(prompt).content

'The distance between Mexico and Thailand depends on the specific locations within each country. However, I can provide you with an approximate distance between the two countries.\n\nMexico City, Mexico (the capital city) to Bangkok, Thailand (the capital city) is approximately:\n\n* 9,541 miles (15,355 kilometers)\n* 8,323 nautical miles\n\nPlease note that this is a straight-line distance, also known as the "great circle distance" or "as the crow flies." The actual distance traveled by air or sea may be longer due to the route taken.\n\nTo give you a better idea, here are some approximate flight distances and times between major cities in Mexico and Thailand:\n\n* Mexico City (MEX) to Bangkok (BKK): 9,541 miles (15,355 km), 18 hours 30 minutes\n* Cancun (CUN) to Bangkok (BKK): 10,141 miles (16,312 km), 20 hours 30 minutes\n* Guadalajara (GDL) to Bangkok (BKK): 9,821 miles (15,802 km), 19 hours 30 minutes\n\nKeep in mind that these distances and flight times are approximate and may va

In [8]:
# 3.2 Prompt Templates
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

chat = ChatOllama(
  model="llama3.3",
  temperature=0.1
)

template = ChatPromptTemplate.from_messages([
  ("system", "You are a geography expert. And you only reply in {language}."),
  ("ai", "Ciao, mi chiamo {name}!"),
  ("human", "What is the distance between {country_a} and {country_b}. Also, what is your name?")
])

prompt = template.format_messages(
  language="Greek",
  name="Socrates",
  country_a="Mexico",
  country_b="Thailand",
)

chat.invoke(prompt).content

'Γεια σας! Ονομάζομαι Γεώργιος και είμαι ειδικός στη γεωγραφία. Η απόσταση μεταξύ του Μεξικού και της Ταϊλάνδης είναι περίπου 17.300 χιλιόμετρα. Αν θέλετε να μάθετε περισσότερα για τις δύο χώρες, είμαι εδώ για να σας βοηθήσω!'

In [9]:
# 3.3 OutputParser and LCEL (LangChain Expression Language)
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import BaseOutputParser

chat = ChatOllama(
  model="llama3.3",
  temperature=0.1
)

class CommaOutputParser(BaseOutputParser):
   
   def parse(self, text):
      item = text.strip().split(",")    # 텍스트의 앞뒤 공백 제거 후 comma(,)로 잘라서 array로 반환 
      return list(map(str.strip, item)) # 각 item에 strip 함수 적용
   
template = ChatPromptTemplate.from_messages([
  ("system", "You are a list generating machine. Evenything you are asked will be answered with a comma separated list of max {max_items} in lowercase. Do NOT reply with anything else."),
  ("human", "{question}"),
])

prompt = template.format_messages(
   max_items=10, 
   question="What are the colors?"
)

result = chat.invoke(prompt)

p = CommaOutputParser()

p.parse(result.content)


['red',
 'blue',
 'green',
 'yellow',
 'orange',
 'purple',
 'pink',
 'brown',
 'grey',
 'black']

In [10]:
# 3.3 OutputParser and LCEL (LangChain Expression Language)
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import BaseOutputParser

chat = ChatOllama(
  model="llama3.3",
  temperature=0.1
)

class CommaOutputParser(BaseOutputParser):
   
   def parse(self, text):
      item = text.strip().split(",")    # 텍스트의 앞뒤 공백 제거 후 comma(,)로 잘라서 array로 반환 
      return list(map(str.strip, item)) # 각 item에 strip 함수 적용
   
template = ChatPromptTemplate.from_messages([
  ("system", "You are a list generating machine. Evenything you are asked will be answered with a comma separated list of max {max_items} in lowercase. Do NOT reply with anything else."),
  ("human", "{question}"),
])

chain = template | chat | CommaOutputParser()

chain.invoke({
   "max_items": 5,
   "question": "What are the pokemons?"
})

['pikachu', 'charizard', 'bulbasaur', 'squirtle', 'charmeleon']

In [11]:
# 3.4 Chaining Chains
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.callbacks import StreamingStdOutCallbackHandler

chat = ChatOllama(
  model="llama3.3",
  temperature=0.1,
  streaming=True,
  callbacks=[StreamingStdOutCallbackHandler()]
)

chef_prompt = ChatPromptTemplate.from_messages([
  ("system", "You are a world-class international chef. You create easy to follow recipies for any type of cuisine with easy to find ingredients."),
  ("human", "I want to cook {cuisine} food.")
])

chef_chain = chef_prompt | chat

veg_chef_prompt = ChatPromptTemplate.from_messages([
  ("system", '''You are a vegetarian chef specialized on making traditional recipies vegetarian. 
   You find alternative ingredients and explain their preparation. 
   You don't radically modify the recipe. 
   If there is no alternative for a food just say you don't know how to replace it.'''),
  ("human", "{recipe}")
])

veg_chain = veg_chef_prompt | chat

final_chain = {"recipe": chef_chain} | veg_chain

final_chain.invoke({
  "cuisine": "indian"
})

Indian cuisine is one of the most diverse and flavorful in the world! With its rich use of spices, herbs, and other ingredients, Indian cooking can be a bit intimidating at first, but don't worry, I'm here to guide you through it.

To get started, let's choose a popular Indian dish that's easy to make and requires minimal special ingredients. How about we make some delicious Chicken Tikka Masala?

**Chicken Tikka Masala Recipe**

 Servings: 4-6 people

**Ingredients:**

For the chicken:

* 1 1/2 pounds boneless, skinless chicken breast or thighs, cut into 1-inch pieces
* 1/2 cup plain yogurt
* 2 tablespoons lemon juice
* 2 teaspoons garam masala powder
* 1 teaspoon cumin powder
* 1/2 teaspoon coriander powder
* 1/2 teaspoon cayenne pepper (optional)
* Salt, to taste

For the sauce:

* 2 large onions, diced
* 2 cloves garlic, minced
* 2 medium tomatoes, diced
* 1 tablespoon butter or ghee
* 2 tablespoons tomato puree
* 1 cup chicken broth
* 1/2 cup heavy cream or half-and-half
* 2 teasp

AIMessage(content='To make a vegetarian version of Chicken Tikka Masala, we can replace the chicken with a plant-based alternative. One option is to use extra-firm tofu, cut into 1-inch pieces, or tempeh. Another option is to use seitan, which is made from wheat gluten and has a chewy texture similar to meat.\n\nHere\'s how you can prepare the vegetarian version:\n\n**For the "chicken":**\n\n* 1 1/2 pounds extra-firm tofu, cut into 1-inch pieces\n* 1/2 cup plain yogurt (you can use a non-dairy yogurt alternative like soy or coconut yogurt)\n* 2 tablespoons lemon juice\n* 2 teaspoons garam masala powder\n* 1 teaspoon cumin powder\n* 1/2 teaspoon coriander powder\n* 1/2 teaspoon cayenne pepper (optional)\n* Salt, to taste\n\nMarinate the tofu in the same way as the chicken, and then grill or bake it until it\'s golden brown and crispy on the outside.\n\nThe rest of the recipe remains the same. You can use the same sauce and spices, and serve it over basmati rice or with naan bread.\n\nAl